# Dashboard evento 13–17 novembre 2026 – WRF meteo@uniparthenope (d03)
Legge il prodotto curato Zarr creato da `caso_uso_wrf_campania.ipynb`.
Aprire con `/user/<utente>/voila/render/notebooks/dashboard_wrf_voila.ipynb`.

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, clear_output

zarr_path = os.environ.get("WRF_ZARR", "/home/jovyan/work/outputs/wrf_campania/wrf_campania_curated.zarr")
ds = xr.open_zarr(zarr_path)
lat, lon, land = ds.lat.values, ds.lon.values, ds.landmask.values
tempi = pd.to_datetime(ds.time.values)
CITTA = {"Napoli": (40.852, 14.268), "Salerno": (40.678, 14.765), "Avellino": (40.914, 14.793),
         "Benevento": (41.130, 14.782), "Caserta": (41.072, 14.333), "Positano": (40.628, 14.485),
         "Capri": (40.551, 14.243)}
CAMPI = {"Pioggia oraria [mm]": ("rain_h", "Blues", 0, 10), "Vento 10 m [m/s]": ("ws10", "YlOrRd", 0, 25),
         "Temperatura 2 m [°C]": ("t2c", "RdYlBu_r", None, None)}

def nearest(la, lo):
    d = (lat - la) ** 2 + ((lon - lo) * np.cos(np.deg2rad(la))) ** 2
    return np.unravel_index(np.argmin(d), d.shape)

ora = w.SelectionSlider(options=[(f"{t:%d/%m %H}:00", i) for i, t in enumerate(tempi)],
                        index=len(tempi) // 2, description="Ora", layout=w.Layout(width="60%"))
campo = w.Dropdown(options=list(CAMPI), description="Campo")
citta = w.Dropdown(options=list(CITTA), value="Positano", description="Città")
out = w.Output()

def aggiorna(*_):
    var, cmap, vmin, vmax = CAMPI[campo.value]
    with out:
        clear_output(wait=True)
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.3]})
        m = a1.pcolormesh(lon, lat, ds[var].isel(time=ora.value), cmap=cmap, vmin=vmin, vmax=vmax,
                          shading="auto")
        a1.contour(lon, lat, land.astype(float), levels=[0.5], colors="k", linewidths=.6)
        la, lo = CITTA[citta.value]; a1.plot(lo, la, "k*", ms=10)
        fig.colorbar(m, ax=a1, label=campo.value); a1.set_aspect("equal")
        a1.set_title(f"{campo.value} – {tempi[ora.value]:%d/%m/%Y %H}:00 UTC")
        j, i = nearest(la, lo)
        s = ds[var].isel(south_north=j, west_east=i)
        if var == "rain_h":
            a2.bar(tempi, s, width=1 / 24, color="tab:blue")
        else:
            a2.plot(tempi, s, color="tab:red")
        a2.axvline(tempi[ora.value], color="k", lw=1)
        a2.set_title(f"{campo.value} – {citta.value}"); a2.grid(alpha=.3)
        fig.autofmt_xdate(); plt.tight_layout(); plt.show()

for c in (ora, campo, citta):
    c.observe(aggiorna, "value")
display(w.VBox([ora, w.HBox([campo, citta]), out]))
aggiorna()